In [2]:
import numpy as np
import pandas as pd
import models
import json
import re
from datetime import datetime
from database import engine, sessionDB
from nba_api.stats.static import players
from nba_api.stats.endpoints import playerawards, commonplayerinfo, playercareerstats
from langchain_ollama import ChatOllama
from decouple import config
from ddgs import DDGS

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
players= players.get_players()
#players

In [5]:
models.Base.metadata.create_all(bind=engine)
db = sessionDB()

player testing and structure allat stuff

In [6]:
#using lebron cus hes the goat
lebron = [p for p in players if p["full_name"].lower() == "lebron james"]
lebron = lebron[0] #de-listifying (dont do this in prod)
pid = lebron["id"] #2544
info_df = commonplayerinfo.CommonPlayerInfo(player_id=pid).get_data_frames()[0] 

needed_columns = [
    "DISPLAY_FIRST_LAST", 
    "BIRTHDATE", 
    "COUNTRY", 
    "HEIGHT", 
    "WEIGHT", 
    "DRAFT_YEAR", 
    "DRAFT_ROUND", 
    "DRAFT_NUMBER"
]

#nicknames and school i need to use an agent for

info_df = info_df[needed_columns]
info_df = info_df.rename({"DISPLAY_FIRST_LAST": "NAME"}, axis=1)

info_df

,NAME,BIRTHDATE,COUNTRY,HEIGHT,WEIGHT,DRAFT_YEAR,DRAFT_ROUND,DRAFT_NUMBER
0,LeBron James,1984-12-30T00:00:00,USA,6-9,250,2003,1,1


actual player info parsing

In [7]:
#height to cm

def to_CM(height: str):
    height = height.split("-")
    return round(int(height[0]) * 30.48 + int(height[1]) * 2.54)

to_CM("6-9")

206

In [8]:
#agent to get nicknames + school

def search(query: str) -> str:
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=5))
    if not results:
        return "no results found"
    return "\n\n".join(
        f"Title: {r['title']}\nSnippet: {r['body']}"
        for r in results
    )

llm = ChatOllama(
    model=config("OLLAMA_MODEL", default="qwen2.5:7b-instruct"), 
    base_url=config("OLLAMA_BASE_URL", "http://localhost:11434/"),
    temperature=0
)

def getExtraPlayerInfo(player_name: str) -> dict:
    search_results = search(f"{player_name} NBA nicknames college university")

    prompt = f"""Based on these search results about NBA player {player_name}:

    {search_results}

    Return ONLY a valid JSON object in this exact format:
    {{"nicknames": ["nickname1", "nickname2"], "college": "University Name or null"}}

    Rules:
    - college should be null if they went straight from high school
    - nicknames should be an empty list if none are found
    - if the nickname is his name then dont include it
    """

    response = llm.invoke(prompt)
    text = response.content

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
        raise ValueError(f"Could not parse response: {text}")


In [ ]:
def getPlayer(pid: int):
    info_df = commonplayerinfo.CommonPlayerInfo(player_id=pid).get_data_frames()[0] 

    needed_columns = [
        "DISPLAY_FIRST_LAST", 
        "BIRTHDATE", 
        "COUNTRY", 
        "HEIGHT", 
        "WEIGHT", 
        "DRAFT_YEAR", 
        "DRAFT_ROUND", 
        "DRAFT_NUMBER"
    ]

    #nicknames and school i need to use an agent for

    info_df = info_df[needed_columns]
    info_df = info_df.rename({"DISPLAY_FIRST_LAST": "NAME"}, axis=1)

    for col in ["DRAFT_YEAR", "DRAFT_ROUND", "DRAFT_NUMBER"]:
        info_df[col] = info_df[col].apply(lambda x: int(x) if str(x).isdigit() else None)
        
    return info_df

bron_id = 2544

info_df = getPlayer(bron_id)
player_name = info_df["NAME"].iloc[0]

extra_info = None
while not extra_info:
    extra_info = getExtraPlayerInfo(player_name)

print(extra_info)

pinfo = models.Player(
    player_id = bron_id, #dont hardcode this in prod
    name = info_df["NAME"].iloc[0],
    nicknames = extra_info['nicknames'],
    country = info_df["COUNTRY"].iloc[0],
    school = extra_info['college'],
    birthdate = datetime.fromisoformat(info_df["BIRTHDATE"].iloc[0]).date(),
    height = to_CM(info_df["HEIGHT"].iloc[0]),
    weight = int(info_df["WEIGHT"].iloc[0]),
    draft_year = int(info_df["DRAFT_YEAR"].iloc[0]) if info_df["DRAFT_YEAR"].iloc[0] else 0,
    draft_round = int(info_df["DRAFT_ROUND"].iloc[0]) if info_df["DRAFT_ROUND"].iloc[0] else 0,
    draft_pick = int(info_df["DRAFT_NUMBER"].iloc[0]) if info_df["DRAFT_NUMBER"].iloc[0] else 0,
)

#db.rollback()
#db.add(pinfo)
#db.commit()
#db.close()


{'nicknames': ['Captain LeMerica', 'King James'], 'college': None}


award testing and structure allat stuff

In [10]:
#using lebron again cus hes the goat
lebron = [p for p in players if p["full_name"].lower() == "lebron james"]
lebron = lebron[0] #de-listifying (dont do this in prod)
lebron_id = lebron["id"] #2544
lebron_awards = playerawards.PlayerAwards(player_id=lebron_id).get_data_frames()[0]

data = lebron_awards

acronym_hmap = {
    "All-Defensive Team": "DEF",''
    "All-NBA": "NBA",
    "All-Rookie Team": "ROOK",
    "NBA All-Star": "AS",
    "NBA All-Star Most Valuable Player": "ASMVP",
    "NBA Champion": "CHAMP",
    "NBA Cup All-Tournament Team": "CUP",
    "NBA Cup Most Valuable Player": "CUPMVP",
    "NBA Finals Most Valuable Player": "FMVP",
    "NBA Most Valuable Player": "MVP",
    "NBA Player of the Month": "POTM",
    "NBA Player of the Week": "POTW",
    "NBA Rookie of the Month": "ROTM",
    "NBA Rookie of the Year": "ROTY",
    "NBA Defensive Player of the Year": "DPOY",
    "NBA Defensive Player of the Month": "DPOM",
    "NBA Most Improved Player": "MIP",
    "NBA Clutch Player of the Year": "CPOY",
    "NBA Sportsmanship": "SPTMAN",
}

all_nba_set = {"DEF", "NBA", "ROOK", "CUP"}

#gotta add stat champs + cfmvp + hustle player oty + teammate oty + dunk/3pt

clean_lebron_awards_df = lebron_awards[lebron_awards["DESCRIPTION"].isin(acronym_hmap)]
clean_lebron_awards_df = clean_lebron_awards_df.drop([
    "FIRST_NAME", 
    "LAST_NAME", 
    "CONFERENCE", 
    "TYPE", 
    "SUBTYPE1", 
    "SUBTYPE2", 
    "SUBTYPE3",
    "MONTH",
    "WEEK",
    "PERSON_ID",
    "TEAM"
], axis=1)

clean_lebron_awards_df["ALL_NBA_TEAM_NUMBER"] = clean_lebron_awards_df["ALL_NBA_TEAM_NUMBER"].fillna(0)
clean_lebron_awards_df = clean_lebron_awards_df.replace(r'^\s*$', 0, regex=True) 
#^ gets rid of empty strings

clean_lebron_awards_df["DESCRIPTION"] = clean_lebron_awards_df["DESCRIPTION"].map(acronym_hmap)
clean_lebron_awards_df["DESCRIPTION"] = clean_lebron_awards_df.apply(
    lambda r: f"{r['DESCRIPTION']}{r['ALL_NBA_TEAM_NUMBER']}" 
    if r['DESCRIPTION'] in all_nba_set else r['DESCRIPTION'], 
    axis=1
)

clean_lebron_awards_df
#data

,DESCRIPTION,ALL_NBA_TEAM_NUMBER,SEASON
0,DEF1,1,2008-09
1,DEF1,1,2009-10
2,DEF1,1,2010-11
3,DEF1,1,2011-12
4,DEF1,1,2012-13
5,DEF2,2,2013-14
6,NBA2,2,2004-05
7,NBA1,1,2005-06
8,NBA2,2,2006-07
9,NBA1,1,2007-08


actual award parsing

In [11]:
def getAwards(pid: int):
    awards_df = playerawards.PlayerAwards(player_id=pid).get_data_frames()[0]

    acronym_hmap = {
        "All-Defensive Team": "DEF",''
        "All-NBA": "NBA",
        "All-Rookie Team": "ROOK",
        "NBA All-Star": "AS",
        "NBA All-Star Most Valuable Player": "ASMVP",
        "NBA Champion": "CHAMP",
        "NBA Cup All-Tournament Team": "CUP",
        "NBA Cup Most Valuable Player": "CUPMVP",
        "NBA Finals Most Valuable Player": "FMVP",
        "NBA Most Valuable Player": "MVP",
        "NBA Player of the Month": "POTM",
        "NBA Player of the Week": "POTW",
        "NBA Rookie of the Month": "ROTM",
        "NBA Rookie of the Year": "ROTY",
    }

    all_nba_set = {"DEF", "NBA", "ROOK", "CUP"}

    awards_df = awards_df[awards_df["DESCRIPTION"].isin(acronym_hmap)]
    awards_df = awards_df.drop([
        "FIRST_NAME", 
        "LAST_NAME", 
        "CONFERENCE", 
        "TYPE", 
        "SUBTYPE1", 
        "SUBTYPE2", 
        "SUBTYPE3",
        "MONTH",
        "WEEK",
        "PERSON_ID",
        "TEAM"
    ], axis=1)

    awards_df["ALL_NBA_TEAM_NUMBER"] = awards_df["ALL_NBA_TEAM_NUMBER"].fillna(0)
    awards_df = awards_df.replace(r'^\s*$', 0, regex=True) 
    #^ gets rid of empty strings

    awards_df["DESCRIPTION"] = awards_df["DESCRIPTION"].map(acronym_hmap)
    awards_df["DESCRIPTION"] = awards_df.apply(
        lambda r: f"{r['DESCRIPTION']}{r['ALL_NBA_TEAM_NUMBER']}" 
        if r['DESCRIPTION'] in all_nba_set else r['DESCRIPTION'], 
        axis=1
    )

    return awards_df.drop(["ALL_NBA_TEAM_NUMBER"], axis=1)

bron_id = 2544

awards = getAwards(bron_id)

award_list = [
    models.Award(                                                   
        player_id = bron_id, #dont hardcode this in prod
        season = award.SEASON,
        award_name = award.DESCRIPTION
    )
    for award in awards.itertuples()
]

#db.rollback()
#db.add_all(award_list)
#db.commit()
#db.close()

season stat testing

In [12]:
lebron = [p for p in players if p["full_name"].lower() == "lebron james"]
lebron = lebron[0] #de-listifying (dont do this in prod)
lebron_id = lebron["id"] #2544

stats = playercareerstats.PlayerCareerStats(player_id=lebron_id).get_data_frames()
regStats = stats[0]
postStats = stats[2]

regStats = regStats.drop([
    "PLAYER_ID",
    "LEAGUE_ID",
], axis=1)

regStats = regStats.rename({"SEASON_ID": "SEASON"}, axis=1)

regStats

,SEASON,TEAM_ID,TEAM_ABBREVIATION,PLAYER_AGE,GP,GS,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS
0,2003-04,1610612739,CLE,19.0,79,79,3120,622,1492,0.417,63,217,0.290,347,460,0.754,99,333,432,465,130,58,273,149,1654
1,2004-05,1610612739,CLE,20.0,80,80,3388,795,1684,0.472,108,308,0.351,477,636,0.750,111,477,588,577,177,52,262,146,2175
2,2005-06,1610612739,CLE,21.0,79,79,3361,875,1823,0.480,127,379,0.335,601,814,0.738,75,481,556,521,123,66,260,181,2478
3,2006-07,1610612739,CLE,22.0,78,78,3190,772,1621,0.476,99,310,0.319,489,701,0.698,83,443,526,470,125,55,250,171,2132
4,2007-08,1610612739,CLE,23.0,75,74,3027,794,1642,0.484,113,359,0.315,549,771,0.712,133,459,592,539,138,81,255,165,2250
5,2008-09,1610612739,CLE,24.0,81,81,3054,789,1613,0.489,132,384,0.344,594,762,0.780,106,507,613,587,137,93,241,139,2304
6,2009-10,1610612739,CLE,25.0,76,76,2966,768,1528,0.503,129,387,0.333,593,773,0.767,71,483,554,651,125,77,261,119,2258
7,2010-11,1610612748,MIA,26.0,79,79,3063,758,1485,0.510,92,279,0.330,503,663,0.759,80,510,590,554,124,50,284,163,2111
8,2011-12,1610612748,MIA,27.0,62,62,2326,621,1169,0.531,54,149,0.362,387,502,0.771,94,398,492,387,115,50,213,96,1683
9,2012-13,1610612748,MIA,28.0,76,76,2877,765,1354,0.565,103,254,0.406,403,535,0.753,97,513,610,551,129,67,226,110,2036


making the actual function to format season stats

In [ ]:
def formatSeasonStats(pid: int):
    stats = playercareerstats.PlayerCareerStats(player_id=pid).get_data_frames()
    regStats = stats[0]
    postStats = stats[2]

    regStats = regStats.drop([
        "PLAYER_ID",
        "LEAGUE_ID",
    ], axis=1)

    postStats = postStats.drop([
        "PLAYER_ID",
        "LEAGUE_ID",
    ], axis=1)

    regStats = regStats.rename({"SEASON_ID": "SEASON"}, axis=1)    
    postStats = postStats.rename({"SEASON_ID": "SEASON"}, axis=1)

    regStats = regStats.fillna(0)
    postStats = postStats.fillna(0)

    object_cols = regStats.select_dtypes(include="object").columns
    regStats[object_cols] = regStats[object_cols].astype(float)

    return regStats, postStats

bron_id = 2544

stats = formatSeasonStats(bron_id)
reg_season = stats[0]
post_season = stats[1]

print(reg_season.dtypes)
post_season.dtypes

'''
reg_season_list = [
    models.SeasonStats(
        player_id   = bron_id,
        season      = row.SEASON,
        team_id     = row.TEAM_ID,
        team_abbr   = row.TEAM_ABBREVIATION,
        season_type = models.SeasonType.regular,
        player_age  = int(row.PLAYER_AGE),
        gp          = row.GP,
        gs          = row.GS,
        minutes     = row.MIN,
        fgm         = row.FGM,
        fga         = row.FGA,
        fg_pct      = row.FG_PCT,
        fg3m        = row.FG3M,
        fg3a        = row.FG3A,
        fg3_pct     = row.FG3_PCT,
        ftm         = row.FTM,
        fta         = row.FTA,
        ft_pct      = row.FT_PCT,
        oreb        = row.OREB,
        dreb        = row.DREB,
        reb         = row.REB,
        ast         = row.AST,
        stl         = row.STL,
        blk         = row.BLK,
        tov         = row.TOV,
        pf          = row.PF,
        pts         = row.PTS,
    )
    for row in reg_season.itertuples()
]

post_season_list = [
    models.SeasonStats(
        player_id   = bron_id,
        season      = row.SEASON,
        team_id     = row.TEAM_ID,
        team_abbr   = row.TEAM_ABBREVIATION,
        season_type = models.SeasonType.playoffs,
        player_age  = int(row.PLAYER_AGE),
        gp          = row.GP,
        gs          = row.GS,
        minutes     = row.MIN,
        fgm         = row.FGM,
        fga         = row.FGA,
        fg_pct      = row.FG_PCT,
        fg3m        = row.FG3M,
        fg3a        = row.FG3A,
        fg3_pct     = row.FG3_PCT,
        ftm         = row.FTM,
        fta         = row.FTA,
        ft_pct      = row.FT_PCT,
        oreb        = row.OREB,
        dreb        = row.DREB,
        reb         = row.REB,
        ast         = row.AST,
        stl         = row.STL,
        blk         = row.BLK,
        tov         = row.TOV,
        pf          = row.PF,
        pts         = row.PTS,
    )
    for row in post_season.itertuples()
]

db.rollback()
db.add_all(reg_season_list)
db.add_all(post_season_list)
db.commit()
db.close()
'''


SEASON                   str
TEAM_ID                int64
TEAM_ABBREVIATION        str
PLAYER_AGE           float64
GP                     int64
GS                     int64
MIN                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int64
PF                     int64
PTS                    int64
dtype: object


'\nreg_season_list = [\n    models.SeasonStats(\n        player_id   = bron_id,\n        season      = row.SEASON,\n        team_id     = row.TEAM_ID,\n        team_abbr   = row.TEAM_ABBREVIATION,\n        season_type = models.SeasonType.regular,\n        player_age  = int(row.PLAYER_AGE),\n        gp          = row.GP,\n        gs          = row.GS,\n        minutes     = row.MIN,\n        fgm         = row.FGM,\n        fga         = row.FGA,\n        fg_pct      = row.FG_PCT,\n        fg3m        = row.FG3M,\n        fg3a        = row.FG3A,\n        fg3_pct     = row.FG3_PCT,\n        ftm         = row.FTM,\n        fta         = row.FTA,\n        ft_pct      = row.FT_PCT,\n        oreb        = row.OREB,\n        dreb        = row.DREB,\n        reb         = row.REB,\n        ast         = row.AST,\n        stl         = row.STL,\n        blk         = row.BLK,\n        tov         = row.TOV,\n        pf          = row.PF,\n        pts         = row.PTS,\n    )\n    for row in reg

In [14]:
history = []
for i in range(10):
    history.append(getExtraPlayerInfo("cooper flagg"))

history

[{'nicknames': [], 'college': 'Duke University'},
 {'nicknames': ['The Maine Event'], 'college': None},
 {'nicknames': [], 'college': None},
 {'nicknames': [], 'college': 'Duke University'},
 {'nicknames': ['The Maine Event'], 'college': 'Duke'},
 {'nicknames': ['The Maine Event'], 'college': 'Duke'},
 {'nicknames': [], 'college': None},
 {'nicknames': ['The Maine Event'], 'college': None},
 {'nicknames': ['The Maine Event'], 'college': None},
 {'nicknames': [], 'college': 'Duke University'}]

misc other testing 

In [34]:
data = getPlayer(76007)
data

,NAME,BIRTHDATE,COUNTRY,HEIGHT,WEIGHT,DRAFT_YEAR,DRAFT_ROUND,DRAFT_NUMBER
0,John Abramovic,1919-02-09T00:00:00,USA,6-3,195,Undrafted,Undrafted,Undrafted
